In [2]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path

%load_ext autoreload
%autoreload 2

# Import the new pipeline
sys.path.append(os.path.abspath(".."))
import src.feature_engineering as feat

pd.set_option('display.max_columns', None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
file_path = Path('../data/processed/ipl_cleaned.csv')
df = pd.read_csv(file_path,low_memory=False)

In [28]:
df = feat.add_basic_event_flags(df) 
# add features like is_wicket, is_legal_delivery, is_dot_ball, is_boundary

# Verify all basic event flags at once
flag_checks = {
    'is_wicket': 'Total Deliveries vs. Total Wickets',
    'is_legal_delivery': 'Legal vs. Illegal Deliveries',
    'is_dot_ball': 'Dot Balls vs Scoring Deliveries',
    'is_boundary': 'Boundaries vs Non-Boundaries'
}

print("--- Event Flag Validations ---")
for col, title in flag_checks.items():
    print(f"\n{title}:")
    print(df[col].value_counts().to_string())

--- Event Flag Validations ---

Total Deliveries vs. Total Wickets:
is_wicket
0    281027
1     14705

Legal vs. Illegal Deliveries:
is_legal_delivery
1    284630
0     11102

Dot Balls vs Scoring Deliveries:
is_dot_ball
0    194502
1    101230

Boundaries vs Non-Boundaries:
is_boundary
0    245506
1     50226


In [29]:
df = feat.add_bowling_team(df)
# add a bowling team feature
df[['team1', 'team2', 'batting_team', 'bowling_team']].head(1)

,team1,team2,batting_team,bowling_team
0,Royal Challengers Bengaluru,Kolkata Knight Riders,Kolkata Knight Riders,Royal Challengers Bengaluru


In [30]:
df = feat.add_match_phases(df)
# add powerplay,middle overs and death overs info

# Verify the distribution of data across the phases
print("--- Deliveries per Match Phase ---")
print(df['phase'].value_counts())

--- Deliveries per Match Phase ---
phase
Middle Overs    135655
Powerplay        92904
Death Overs      67173
Name: count, dtype: int64


In [31]:
df = feat.calculate_cumulative_scores(df)
#calculates running total for scores and wickets per inning

In [32]:
df = feat.add_run_chase_features(df)

# add a run chase and balls remaining feature for second innings (this will be NaN for inning 1)

In [33]:
df = feat.flag_rain_reduced_matches(df)
# add flag for matches interrupted by rain (DLS Method)

Total Rain-Reduced Matches Flagged: 3630


In [35]:
print(df['innings'].unique())

[1 2 3 4 5 6]


In [36]:
# Super Over Split
# Filter for standard T20 innings (1 and 2)
regular_df = df[df['innings'] <= 2].copy()

# Filter for Super Overs (Innings 3, 4, etc.)
super_overs_df = df[df['innings'] > 2].copy()


# --- THE EXPORT ---
output_dir = Path('../data/processed')

# Define the two separate file paths
regular_file = output_dir / 'ipl_features_regular.csv'
super_over_file = output_dir / 'ipl_features_super_overs.csv'

# Save both DataFrames
regular_df.to_csv(regular_file, index=False)
super_overs_df.to_csv(super_over_file, index=False)

print(f"Standard Match Data saved: {len(regular_df)} rows")
print(f"Super Over Data saved: {len(super_overs_df)} rows")

Standard Match Data saved: 295557 rows
Super Over Data saved: 175 rows
